# ניתוח תביעת הוצאות

המחברת הזאת מדגימה איך ליצור סוכנים שמשתמשים בתוספים לעיבוד הוצאות טיול מתמונות חשבוניות מקומיות, ליצירת מייל תביעה להחזר הוצאות, ולהמחשת נתוני הוצאות באמצעות תרשים עוגה. הסוכנים בוחרים פונקציות דינמית בהתאם להקשר המשימה.

שלבים:
1. סוכן OCR מעבד את תמונת החשבונית המקומית ומחלץ נתוני הוצאות טיול.
2. סוכן דואר אלקטרוני יוצר מייל תביעה להחזר הוצאות.

### דוגמה לתרחיש הוצאות טיול:
דמיין שאתה עובד שנוסע לפגישה עסקית בעיר אחרת. לחברה שלך יש מדיניות להחזיר את כל ההוצאות הסבירות הקשורות לנסיעה. הנה פירוט של הוצאות טיול אפשריות:
- תחבורה:
כרטיס טיסה הלוך ושוב מעיר המגורים שלך לעיר היעד.
מונית או שירותי הזמנת נסיעה הלוך ושוב לנמל התעופה.
תחבורה מקומית בעיר היעד (כגון תחבורה ציבורית, השכרת רכב או מוניות).

- לינה:
שהייה במלון לשלוש לילות במלון עסקי ברמת ביניים קרוב למקום הפגישה.

- ארוחות:
סכום יומי לארוחות ארוחת בוקר, צהריים וערב, בהתבסס על מדיניות דמי היוממות של החברה.

- הוצאות שונות:
דמי חנייה בנמל התעופה.
עלויות גישה לאינטרנט במלון.
טיפים או חיובי שירות קטנים.

- תיעוד:
אתה מגיש את כל החשבוניות (טיסות, מוניות, מלון, ארוחות וכו’) ודוח הוצאות מלא להחזר.


## יבוא ספריות נדרשות

ייבא את הספריות והמודולים הדרושים לפנקס המחברת.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import Content, Message, tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## הגדרת דגמי הוצאה

 צור מודל פידנטיק להוצאות בודדות ומחלקת ExpenseFormatter להמרת שאילתת משתמש לנתוני הוצאה מובנים.

 כל הוצאה תיוצג בפורמט הבא:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## הגדרת כלים - יצירת דואר אלקטרוני

צור פונקציית כלי ליצירת דואר אלקטרוני לצורך הגשת תביעת הוצאות.
- כלי זה משתמש בדקורטור `@tool` ממסגרת העבודה של Microsoft Agent.
- הוא מחשב את הסכום הכולל של ההוצאות ומעצב את הפרטים לתוך גוף הדואר האלקטרוני.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# טעינת תמונות קבלות הוצאות נסיעה

צור פונקציית עזר לטעינת תמונות קבלה כתוכן מולטימודלי מקומי.
- היא קוראת את בתיי התמונה של הקבלה ועוטפת אותם במסגרת הסוכן `Content` כדי שהסוכן יוכל לנתח אותם.


In [ ]:
def load_receipt_image(image_path: str = "receipt.jpg") -> Content:
    """Load a receipt image as native multimodal content."""
    with open(image_path, "rb") as f:
        image_bytes = f.read()
    return Content.from_data(image_bytes, "image/jpeg")

## עיבוד הוצאות

הגדירו את הסוכנים וחברו אותם לזרימת עבודה סדרתית באמצעות `WorkflowBuilder`.
- סוכן ה-OCR מחלץ נתוני הוצאה מובנים מתמונת הקבלה שסופקה בהודעת המשתמש.
- סוכן המייל לוקח את הנתונים שחולצו ויוצר מייל תביעת הוצאה מקצועי באמצעות הכלי `generate_expense_email`.
- `WorkflowBuilder` עם `add_edge` יוצרים צינור סדרתי: סוכן OCR → סוכן מייל.


In [ ]:
ocr_agent = client.as_agent(
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Analyze the receipt image supplied in the user message and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## הפונקציה הראשית

בנה את זרימת העבודה הסדרתית והפעל אותה לעיבוד תמונת הקבלה וליצירת דואר טענת ההוצאה.


> **הערה:** בתים של הקבלה עטופים במגוון המצבים של מסגרת סוכן מקומית `Content`, ולכן Foundry מקבל אותם כקלט תמונה במקום טקסט רגיל.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
receipt_message = Message(
    role="user",
    contents=[prompt, load_receipt_image("receipt.jpg")],
)

events = workflow.run(
    receipt_message,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
